# 6. Immutability and provenance

A `PropertyMap` is frozen. Every operation that would change it returns a new
map instead, and the arrays it exposes are read-only. This chapter shows what
that buys you, and what a map remembers about its own construction.

In [ ]:
from summer4 import Property, PropertyMap, Stratification

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
severity = Property("severity", ("mild", "severe"))

base = PropertyMap.from_property(state)
pmap = base.stratify(age).stratify(severity, where=state["I"])

## Nothing is modified in place

`stratify` is a pure function of the map. Intermediate maps stay valid, so a
script can branch: build a common core, then explore two stratification
strategies without rebuilding.

In [ ]:
core = base.stratify(age)

with_severity = core.stratify(severity, where=state["I"])
with_vaccination = core.stratify(Property("vaccination", ("none", "dose")))

assert core.size == 9
assert with_severity.size == 12
assert with_vaccination.size == 18

## The code table is read-only

`codes` is exposed for inspection and for the array work that a later solver
layer will do, but it cannot be written through.

In [ ]:
import numpy as np

assert pmap.codes.dtype == np.int16
assert not pmap.codes.flags.writeable

try:
    pmap.codes[0, 0] = 2
except ValueError as exc:
    print(exc)

## Provenance: `history`

Every map records the stratifications that produced it, in order, including the
selector each was restricted to. This is the audit trail for "how did this
compartment space come to exist?".

In [ ]:
assert len(pmap.history) == 2

for step in pmap.history:
    where = "everything" if step.where is None else step.where
    print(f"{step.property.name:<10} traits={step.property.traits}  where={where}")

Because `Stratification` is a value, history entries can be replayed onto
another map. This is how a documented model structure can be reused rather than
copy-pasted.

In [ ]:
replayed = PropertyMap.from_property(state)
for step in pmap.history:
    replayed = step.apply(replayed)

assert replayed == pmap

## Provenance: `parent_row`

`parent_row[i]` is the row of the *previous* map that compartment `i` was
expanded from. It is the mapping you need to redistribute an initial population
across a new stratification, or to aggregate a stratified result back to the
coarser space.

In [ ]:
parent = np.asarray(pmap.parent_row)
assert parent.shape == (pmap.size,)

before = core.labels()
for index, (label, source) in enumerate(zip(pmap.labels(), parent)):
    print(f"{index:>3}  {label:<30} <- {before[source]}")

### Using `parent_row` to split a population

Splitting a coarse population vector across the finest stratification is a
gather followed by a division by the group size.

In [ ]:
coarse = np.full(core.size, 300.0)          # 300 people in each of 9 compartments
counts = np.bincount(parent, minlength=core.size)
fine = coarse[parent] / counts[parent]

assert abs(fine.sum() - coarse.sum()) < 1e-9
assert fine[pmap.select(state["I"])].tolist() == [150.0] * 6  # split in two

```{admonition} Not yet an API
:class: note

The split above is written out by hand on purpose. summer4 has no
`set_initial_population` / `adjust_population_split` equivalent yet; see
{doc}`../evaluation/feature-completeness`.
```

## Equality and copying

Maps compare by value: same properties, same history, same codes, same parent
rows. `copy()` produces an equal map with a fresh query cache.

In [ ]:
rebuilt = (
    PropertyMap.from_property(state).stratify(age).stratify(severity, where=state["I"])
)
assert rebuilt == pmap

duplicate = pmap.copy()
assert duplicate == pmap
assert duplicate is not pmap

```{admonition} Maps are not hashable
:class: warning

`PropertyMap` defines `__eq__` without `__hash__`, so a map cannot be a
dictionary key or a JAX pytree auxiliary value. Making maps hashable by content
digest is an explicit prerequisite for the JAX work — see
{doc}`../dev/explorations`.
```

In [ ]:
try:
    {pmap: "value"}
except TypeError as exc:
    print(exc)

---

That is the whole of the current public API. {doc}`07-from-summer2` maps it back
to the summer2 vocabulary, and lists what has no equivalent yet.